In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colormaps
import matplotlib

# All coefficient computation is imported from the package.
from polar_express import optimal_composition

matplotlib.rcParams.update({
    "font.family":       "serif",
    "mathtext.fontset":  "cm",
    "font.size":         12,
    "axes.titlesize":    13,
    "axes.labelsize":    12,
    "legend.fontsize":   11,
    "lines.linewidth":   2.5,
    "axes.linewidth":    1.2,
    "axes.grid":         True,
    "grid.linestyle":    "--",
    "grid.linewidth":    0.6,
    "grid.alpha":        0.35,
    "legend.framealpha": 0.95,
    "figure.dpi":        120,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

def eval_poly(coeffs, x):
    """Evaluate odd polynomial  c0·x + c1·x³ + …  at scalar or array x."""
    return sum(c * x**(2*i + 1) for i, c in enumerate(coeffs))

In [ ]:
def plot_composition_arrows(l1, n_steps=12, degree=5, figsize=(16, 4.8)):
    """
    Viridis colour-graded sequence of polynomials with arrows that trace
    the evolution of the lower bound ℓ_t → ℓ_{t+1}.

    Uses optimal_composition from the package.
    """
    coeffs_seq = optimal_composition(
        l1, n_steps, degree=degree, safety_factor_eps=0.01, cushion=0.02
    )
    fig, ax = plt.subplots(figsize=figsize)
    cmap   = colormaps["viridis"]
    x_grid = np.linspace(0, 1, 500)
    l_cur  = float(l1)

    for i, coeffs in enumerate(coeffs_seq):
        color  = cmap(i / max(1, len(coeffs_seq) - 1))
        l_next = float(eval_poly(coeffs, l_cur))

        ax.plot(x_grid, eval_poly(coeffs, x_grid), color=color, linewidth=3.5,
                alpha=0.85, label=r'$\ell={:.3f}$'.format(l_cur))

        # Horizontal arrow: (ℓ_cur, ℓ_next) → (ℓ_next, ℓ_next)
        ax.annotate('', xy=(l_next, l_next), xytext=(l_cur, l_next),
                    arrowprops=dict(arrowstyle='->'))
        # Vertical arrow: (ℓ_cur, ℓ_cur) → (ℓ_cur, ℓ_next)
        if i > 0:
            ax.annotate('', xy=(l_cur, l_next), xytext=(l_cur, l_cur),
                        arrowprops=dict(arrowstyle='->'))

        ax.scatter([l_cur], [l_next], color='red', s=50, zorder=5)
        l_cur = l_next

    ax.plot(x_grid, x_grid, 'k--', linewidth=0.5)
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_xlabel(r'$x$',    fontsize=14)
    ax.set_ylabel(r'$p(x)$', fontsize=14)
    ax.legend(loc='upper left', fontsize=11)
    ax.tick_params(axis='both', labelsize=12)
    plt.tight_layout()
    return fig

fig = plot_composition_arrows(l1=0.001, n_steps=12, degree=5)
plt.show()